# Узлы графа из SAM/SAM2

Построение узлов графа диаграммы из сегментации SAM/SAM2 (с прогревом кэша сегментов).

In [ ]:
from pathlib import Path
import os
import sys

def _find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'vqa_retrieval').exists() and (candidate / 'experiments').exists():
            return candidate
    raise RuntimeError('Cannot find ai2d_vqa_clean root')

PROJECT_ROOT = _find_project_root()
EXTERNAL_ROOT = PROJECT_ROOT.parent
AI2D_ROOT = EXTERNAL_ROOT / 'ai2d'
DOCVQA_ROOT = EXTERNAL_ROOT / 'docvqa'
INFOGRAPHICVQA_ROOT = EXTERNAL_ROOT / 'infographicvqa'
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('AI2D_ROOT =', AI2D_ROOT)


# 16. SAM/SAM2 Graph Nodes with Cache Warmup

Goal: improve diagram graph nodes by replacing or augmenting OpenCV contours with SAM/SAM2 segmentation boxes.

This notebook is safe to run without SAM installed: OpenCV fallback works, SAM sections report what is missing. The important design point is **Stage 0 cache warmup**: segmentation is computed once and reused during training.


## What is needed

For SAM v1:

```powershell
pip install segment-anything
```

Recommended checkpoint location:

```text
../models/sam/sam_vit_b_01ec64.pth
```

For SAM2:

```powershell
pip install sam2
```

Recommended checkpoint directory:

```text
../models/sam2/
```

Cache output used in this notebook:

```text
runs/sam_cache_ai2d/
```


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional, Sequence

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch_geometric.data import Data, Batch

from vqa_retrieval.graph_builder_v2 import (
    NodeV2,
    NodeFeaturizerV2,
    build_typed_edges_v2,
    detect_shapes_opencv_v2,
    parse_ocr_v2_json,
)

HAS_SEGMENT_ANYTHING = importlib.util.find_spec('segment_anything') is not None
HAS_SAM2 = importlib.util.find_spec('sam2') is not None
SAM_CHECKPOINT = EXTERNAL_ROOT / 'models' / 'sam' / 'sam_vit_b_01ec64.pth'
SAM_CACHE_DIR = PROJECT_ROOT / 'runs' / 'sam_cache_ai2d'
SAM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('segment_anything:', HAS_SEGMENT_ANYTHING)
print('sam2:', HAS_SAM2)
print('sam checkpoint:', SAM_CHECKPOINT, 'exists=', SAM_CHECKPOINT.exists())
print('cache:', SAM_CACHE_DIR)


## Load sample image and OCR

Use `3.png` because it is easy to inspect and matches the open annotation tab. You can change `IMAGE_ID` later.


In [ ]:
IMAGE_ID = '3'
IMAGE_PATH = AI2D_ROOT / 'images' / f'{IMAGE_ID}.png'
OCR_PATH = AI2D_ROOT / 'prepared_v2' / 'ocr_v2' / f'{IMAGE_ID}.ocr.json'
ANN_PATH = AI2D_ROOT / 'annotations' / f'{IMAGE_ID}.png.json'

print('image:', IMAGE_PATH, IMAGE_PATH.exists())
print('ocr:', OCR_PATH, OCR_PATH.exists())
print('annotation:', ANN_PATH, ANN_PATH.exists())

pil_img = Image.open(IMAGE_PATH).convert('RGB')
img_bgr = cv2.imread(str(IMAGE_PATH))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
ocr_nodes = parse_ocr_v2_json(OCR_PATH, level='line', min_conf=35.0) if OCR_PATH.exists() else []
print('ocr nodes:', len(ocr_nodes))
plt.figure(figsize=(7, 7))
plt.imshow(img_rgb)
plt.axis('off');


## OpenCV baseline nodes


In [ ]:
def draw_boxes(image_rgb, boxes, title, color='red', max_boxes=80):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image_rgb)
    for bbox in boxes[:max_boxes]:
        x1, y1, x2, y2 = bbox
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(f'{title}: {len(boxes)} boxes')
    ax.axis('off')
    return fig, ax

opencv_boxes = detect_shapes_opencv_v2(img_bgr, min_area=300, max_nodes=80)
print('opencv boxes:', len(opencv_boxes))
draw_boxes(img_rgb, opencv_boxes, 'OpenCV baseline')


## SAM/SAM2 segment extraction

The notebook supports two modes:

- If `segment_anything` and checkpoint are available: run SAM and save masks as bbox segments.
- Otherwise: use OpenCV boxes as a deterministic fallback so the rest of the cache and graph pipeline remains runnable.


In [ ]:
@dataclass(frozen=True)
class SamSegment:
    bbox: tuple[int, int, int, int]
    area: int
    predicted_iou: float = 0.0
    stability_score: float = 0.0
    source: str = 'opencv_fallback'

    def to_dict(self):
        payload = asdict(self)
        payload['bbox'] = list(self.bbox)
        return payload


def mask_to_bbox(mask: np.ndarray) -> Optional[tuple[int, int, int, int]]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0, x2-x1) * max(0, y2-y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = bbox_area((ix1, iy1, ix2, iy2))
    union = bbox_area(a) + bbox_area(b) - inter
    return 0.0 if union <= 0 else inter / union


def deduplicate_boxes_iou(segments: Sequence[SamSegment], iou_threshold: float = 0.85) -> list[SamSegment]:
    out: list[SamSegment] = []
    for seg in sorted(segments, key=lambda s: s.area, reverse=True):
        if all(bbox_iou(seg.bbox, prev.bbox) < iou_threshold for prev in out):
            out.append(seg)
    return out


def filter_segment_boxes(segments, min_area=120, max_area_ratio=0.75, image_size=None, max_segments=120):
    if image_size is None:
        image_area = 1
    else:
        w, h = image_size
        image_area = max(1, w*h)
    filtered = [s for s in segments if s.area >= min_area and s.area <= image_area * max_area_ratio]
    return deduplicate_boxes_iou(filtered)[:max_segments]


def detect_segments_opencv_fallback(image_bgr, min_area=300, max_nodes=80):
    boxes = detect_shapes_opencv_v2(image_bgr, min_area=min_area, max_nodes=max_nodes)
    return [SamSegment(bbox=tuple(map(int, b)), area=bbox_area(b), source='opencv_fallback') for b in boxes]


def detect_segments_sam(image_rgb, checkpoint=SAM_CHECKPOINT, model_type='vit_b', device=None):
    if not HAS_SEGMENT_ANYTHING or not Path(checkpoint).exists():
        print('SAM unavailable. Using OpenCV fallback. Install segment-anything and place checkpoint at:', checkpoint)
        return detect_segments_opencv_fallback(cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
    from segment_anything import SamAutomaticMaskGenerator, sam_model_registry
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    sam = sam_model_registry[model_type](checkpoint=str(checkpoint)).to(device)
    generator = SamAutomaticMaskGenerator(sam)
    masks = generator.generate(image_rgb)
    segments = []
    for item in masks:
        bbox = mask_to_bbox(item.get('segmentation'))
        if bbox is None:
            continue
        segments.append(SamSegment(
            bbox=bbox,
            area=int(item.get('area', bbox_area(bbox))),
            predicted_iou=float(item.get('predicted_iou', 0.0)),
            stability_score=float(item.get('stability_score', 0.0)),
            source=f'sam_{model_type}',
        ))
    return filter_segment_boxes(segments, image_size=Image.fromarray(image_rgb).size)

sam_segments = filter_segment_boxes(detect_segments_sam(img_rgb), image_size=pil_img.size)
print('segments:', len(sam_segments), 'source examples:', sorted({s.source for s in sam_segments})[:5])
draw_boxes(img_rgb, [s.bbox for s in sam_segments], 'SAM/SAM2 or OpenCV fallback', color='lime')


## Stage 0: SAM cache warmup

This is the important training-time step. Build segmentation cache once, then graph construction loads cached segments instead of recomputing SAM inside each batch.


In [ ]:
def image_id_from_path(image_path: Path) -> str:
    return Path(image_path).stem


def sam_cache_path(image_path: Path, cache_dir: Path = SAM_CACHE_DIR) -> Path:
    return cache_dir / f'{image_id_from_path(image_path)}.sam_segments.json'


def load_sam_cache(image_path: Path, cache_dir: Path = SAM_CACHE_DIR) -> Optional[dict[str, Any]]:
    path = sam_cache_path(image_path, cache_dir)
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding='utf-8'))
    try:
        if payload.get('image_mtime_ns') != Path(image_path).stat().st_mtime_ns:
            return None
    except FileNotFoundError:
        return None
    return payload


def save_sam_cache(image_path: Path, segments: Sequence[SamSegment], cache_dir: Path = SAM_CACHE_DIR, backend='sam_vit_b_or_fallback') -> Path:
    path = sam_cache_path(image_path, cache_dir)
    payload = {
        'image_path': str(Path(image_path).resolve()),
        'image_id': image_id_from_path(image_path),
        'sam_backend': backend,
        'checkpoint': str(SAM_CHECKPOINT),
        'image_mtime_ns': Path(image_path).stat().st_mtime_ns,
        'segments': [s.to_dict() for s in segments],
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return path


def build_or_load_sam_cache(image_path: Path, cache_dir: Path = SAM_CACHE_DIR, missing='build') -> dict[str, Any]:
    cached = load_sam_cache(image_path, cache_dir)
    if cached is not None:
        return cached
    if missing == 'skip':
        segments = detect_segments_opencv_fallback(cv2.imread(str(image_path)))
    elif missing == 'build':
        rgb = np.array(Image.open(image_path).convert('RGB'))
        segments = filter_segment_boxes(detect_segments_sam(rgb), image_size=Image.open(image_path).size)
    else:
        raise ValueError("missing must be 'build' or 'skip'")
    save_sam_cache(image_path, segments, cache_dir=cache_dir)
    return load_sam_cache(image_path, cache_dir)

cache_payload = build_or_load_sam_cache(IMAGE_PATH, missing='build')
print('cache file:', sam_cache_path(IMAGE_PATH))
print('cached segments:', len(cache_payload['segments']))
print(json.dumps({k: cache_payload[k] for k in ['image_id', 'sam_backend', 'checkpoint']}, indent=2, ensure_ascii=False))


## Build graph from SAM cache


In [ ]:
def sam_segments_to_nodes(cache_payload: dict[str, Any]) -> list[NodeV2]:
    nodes = []
    for item in cache_payload.get('segments', []):
        bbox = tuple(int(v) for v in item['bbox'])
        nodes.append(NodeV2(bbox=bbox, kind='segment'))
    return nodes


def build_sam_graph_from_cache(image_path: Path, ocr_path: Optional[Path], featurizer: NodeFeaturizerV2, cache_payload: dict[str, Any]) -> Data:
    pil = Image.open(image_path).convert('RGB')
    segment_nodes = sam_segments_to_nodes(cache_payload)
    text_nodes = parse_ocr_v2_json(ocr_path, level='line', min_conf=35.0) if ocr_path and Path(ocr_path).exists() else []
    nodes = segment_nodes + text_nodes
    edge_index, edge_type = build_typed_edges_v2(nodes, k=4)
    x = featurizer.extract(pil, nodes)
    data = Data(x=x, edge_index=edge_index, edge_type=edge_type)
    data.num_segment_nodes = len(segment_nodes)
    data.num_text_nodes = len(text_nodes)
    return data

featurizer = NodeFeaturizerV2(device='cpu')
sam_graph = build_sam_graph_from_cache(IMAGE_PATH, OCR_PATH, featurizer, cache_payload)
print('x:', tuple(sam_graph.x.shape))
print('edge_index:', tuple(sam_graph.edge_index.shape))
print('edge_type:', tuple(sam_graph.edge_type.shape))
print('segment/text:', sam_graph.num_segment_nodes, sam_graph.num_text_nodes)


## Compare OpenCV-only, SAM-only, and late fusion prototype


In [ ]:
def build_opencv_graph(image_path: Path, ocr_path: Optional[Path], featurizer: NodeFeaturizerV2) -> Data:
    pil = Image.open(image_path).convert('RGB')
    bgr = cv2.imread(str(image_path))
    shape_nodes = [NodeV2(bbox=b, kind='shape') for b in detect_shapes_opencv_v2(bgr, min_area=300, max_nodes=80)]
    text_nodes = parse_ocr_v2_json(ocr_path, level='line', min_conf=35.0) if ocr_path and Path(ocr_path).exists() else []
    nodes = shape_nodes + text_nodes
    edge_index, edge_type = build_typed_edges_v2(nodes, k=4)
    x = featurizer.extract(pil, nodes)
    data = Data(x=x, edge_index=edge_index, edge_type=edge_type)
    data.num_shape_nodes = len(shape_nodes)
    data.num_text_nodes = len(text_nodes)
    return data

class TinyGraphEmbedder(nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))

    def forward(self, data: Data):
        return F.normalize(self.proj(data.x).mean(dim=0, keepdim=True), dim=-1)

class LateFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fusion = nn.Sequential(nn.Linear(dim * 2, dim), nn.GELU(), nn.Linear(dim, dim))

    def forward(self, z_opencv, z_sam):
        return F.normalize(self.fusion(torch.cat([z_opencv, z_sam], dim=-1)), dim=-1)

opencv_graph = build_opencv_graph(IMAGE_PATH, OCR_PATH, featurizer)
embedder = TinyGraphEmbedder(opencv_graph.x.size(1), out_dim=64)
fusion = LateFusion(dim=64)
z_opencv = embedder(opencv_graph)
z_sam = embedder(sam_graph)
z_late = fusion(z_opencv, z_sam)

summary = {
    'opencv_nodes': int(opencv_graph.x.size(0)),
    'sam_nodes': int(sam_graph.x.size(0)),
    'z_opencv_shape': list(z_opencv.shape),
    'z_sam_shape': list(z_sam.shape),
    'z_late_shape': list(z_late.shape),
}
print(json.dumps(summary, indent=2))
out = PROJECT_ROOT / 'runs' / 'sam_graph_nodes_smoke' / 'summary.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', out)
